# Hito 1 — Baseline Notebook
**F1 Race Strategy Advisor | Capstone Module 5 — Unit IV**

| Locked decision | Value |
|----------------|-------|
| Target | `is_top10` |
| Train | 2019, 2020, 2021 |
| Calibration | 2022 |
| Test | 2023, 2024 |
| Reference floor (grid-rule) | Brier 0.208 |
| Reference target (docent) | Brier 0.132 / ROC-AUC 0.892 |

## 0 — Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

np.random.seed(42)
print('Libraries loaded. sklearn:', __import__('sklearn').__version__)

Libraries loaded. sklearn: 1.6.1


## 1 — Load Dataset

Dataset oficial del curso: `f1_strategy_race_level.csv` (un registro por piloto por carrera, 2019–2024).

In [2]:
df = pd.read_csv('data/f1_strategy_race_level.csv')
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} columns')
print('Seasons:', sorted(df['season'].unique()))
print('Columns:', list(df.columns))
df.head(3)

Loaded: 2447 rows x 47 columns
Seasons: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Columns: ['season', 'round', 'race_name', 'circuit_id', 'circuit', 'circuit_type', 'driver_id', 'driver_name', 'Driver', 'Team', 'constructor_name', 'grid_position', 'qualifying_position', 'qualifying_time_s', 'driver_prior3_avg_finish', 'constructor_prior3_avg_finish', 'driver_circuit_prior_avg', 'constructor_tier', 'n_stops', 'strategy_type', 'compound_sequence', 'stint_lengths', 'stint1_length', 'stint2_length', 'stint3_length', 'stint4_length', 'stint5_length', 'avg_pit_stop_duration_s', 'total_pit_time_s', 'first_pit_lap', 'last_pit_lap', 'track_status_summary', 'safety_car_periods', 'safety_car_laps', 'vsc_laps', 'weather_actual', 'wet_laps', 'avg_track_temp', 'avg_air_temp', 'finish_position', 'points', 'positions_gained', 'is_top3', 'is_top5', 'is_top10', 'dnf', 'status']


,season,round,race_name,circuit_id,circuit,circuit_type,driver_id,driver_name,Driver,Team,...,avg_track_temp,avg_air_temp,finish_position,points,positions_gained,is_top3,is_top5,is_top10,dnf,status
0,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,bottas,Bottas,BOT,Mercedes,...,40.300000,23.329091,1,26.0,1.0,1,1,1,0,Finished
1,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,hamilton,Hamilton,HAM,Mercedes,...,40.260000,23.330909,2,18.0,-1.0,1,1,1,0,Finished
2,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,max_verstappen,Verstappen,VER,Red Bull,...,40.276364,23.334545,3,15.0,1.0,1,1,1,0,Finished


In [3]:
print('Target:', df['is_top10'].value_counts().to_dict())
print(f'Base rate: {df["is_top10"].mean():.3f}')
print('constructor_tier:', df['constructor_tier'].value_counts().to_dict())
print('strategy_type:', df['strategy_type'].value_counts().to_dict())
print('qualifying_time_s nulls:', df['qualifying_time_s'].isnull().sum(), '— dataset limitation: qualifying time unavailable')

Target: {1: 1263, 0: 1184}
Base rate: 0.516
constructor_tier: {'midfield': 1218, 'backmarker': 780, 'front': 449}
strategy_type: {'one_stop': 1069, 'two_stop': 942, 'three_plus_stop': 376, 'no_stop': 60}
qualifying_time_s nulls: 2447 — dataset limitation: qualifying time unavailable


## 2 — Leakage Audit

Cada columna clasificada como:
- **PRE-RACE**: conocida antes de la carrera — predictores válidos
- **SCENARIO INPUT**: observación post-carrera usada **intencionalmente** como variable what-if (permitido en este capstone)
- **TARGET**: etiqueta a predecir
- **AUDIT / STRESS-TEST**: resultado de carrera que NO debe usarse como predictor silencioso
- **POST-RACE OUTCOME**: excluir del modelado
- **IDENTIFIER**: claves, no features

In [4]:
LEAKAGE_AUDIT = [
    ('season','IDENTIFIER'), ('round','IDENTIFIER'), ('race_name','IDENTIFIER'),
    ('circuit_id','IDENTIFIER'), ('circuit','IDENTIFIER'), ('driver_id','IDENTIFIER'),
    ('driver_name','IDENTIFIER'), ('Driver','IDENTIFIER'), ('Team','IDENTIFIER'),
    ('constructor_name','IDENTIFIER'),
    ('grid_position','PRE-RACE — posicion oficial en la parrilla'),
    ('qualifying_position','PRE-RACE — proxy de grid_position (ver limitacion §2)'),
    ('qualifying_time_s','PRE-RACE — TODO NULL en este dataset (limitacion conocida)'),
    ('constructor_tier','PRE-RACE — front / midfield / backmarker asignado por carrera'),
    ('circuit_type','PRE-RACE — permanent / street / semi-street'),
    ('driver_prior3_avg_finish','PRE-RACE — promedio ultimas 3 carreras'),
    ('constructor_prior3_avg_finish','PRE-RACE — promedio equipo ultimas 3 carreras'),
    ('driver_circuit_prior_avg','PRE-RACE — historial del piloto en este circuito'),
    ('n_stops','SCENARIO INPUT — ingeniero lo fija para comparar estrategias'),
    ('strategy_type','SCENARIO INPUT — one_stop / two_stop / three_plus_stop'),
    ('compound_sequence','SCENARIO INPUT — e.g. S-M-H'),
    ('stint_lengths','SCENARIO INPUT'), ('stint1_length','SCENARIO INPUT'),
    ('stint2_length','SCENARIO INPUT'), ('stint3_length','SCENARIO INPUT'),
    ('stint4_length','SCENARIO INPUT'), ('stint5_length','SCENARIO INPUT'),
    ('avg_pit_stop_duration_s','SCENARIO INPUT — post-carrera'),
    ('total_pit_time_s','SCENARIO INPUT — post-carrera'),
    ('first_pit_lap','SCENARIO INPUT — post-carrera'),
    ('last_pit_lap','SCENARIO INPUT — post-carrera'),
    ('safety_car_periods','AUDIT SLICE — resultado de carrera; solo stress-test, NO predictor'),
    ('safety_car_laps','AUDIT SLICE'), ('vsc_laps','AUDIT SLICE'),
    ('weather_actual','AUDIT SLICE — no conocido con certeza antes de la carrera'),
    ('wet_laps','AUDIT SLICE'), ('avg_track_temp','AUDIT SLICE'),
    ('avg_air_temp','AUDIT SLICE'), ('track_status_summary','AUDIT SLICE'),
    ('is_top10','TARGET — 1 si finish_position <= 10'),
    ('finish_position','POST-RACE OUTCOME — usado para construir target'),
    ('points','POST-RACE OUTCOME'), ('positions_gained','POST-RACE OUTCOME'),
    ('is_top3','POST-RACE OUTCOME'), ('is_top5','POST-RACE OUTCOME'),
    ('dnf','POST-RACE OUTCOME'), ('status','POST-RACE OUTCOME'),
]
audit_df = pd.DataFrame(LEAKAGE_AUDIT, columns=['Column','Classification'])
print(audit_df.to_string(index=False))

                       Column                                                     Classification
                       season                                                         IDENTIFIER
                        round                                                         IDENTIFIER
                    race_name                                                         IDENTIFIER
                   circuit_id                                                         IDENTIFIER
                      circuit                                                         IDENTIFIER
                    driver_id                                                         IDENTIFIER
                  driver_name                                                         IDENTIFIER
                       Driver                                                         IDENTIFIER
                         Team                                                         IDENTIFIER
             constructor_name 

## 3 — Temporal Split

In [5]:
TRAIN_SEASONS = [2019, 2020, 2021]
CALIB_SEASONS = [2022]
TEST_SEASONS  = [2023, 2024]

train = df[df['season'].isin(TRAIN_SEASONS)].copy()
calib = df[df['season'].isin(CALIB_SEASONS)].copy()
test  = df[df['season'].isin(TEST_SEASONS)].copy()

print(f'Train  (2019-2021): {len(train):4d} rows | is_top10={train["is_top10"].mean():.3f}')
print(f'Calib  (2022):      {len(calib):4d} rows | is_top10={calib["is_top10"].mean():.3f}')
print(f'Test   (2023-2024): {len(test):4d} rows  | is_top10={test["is_top10"].mean():.3f}')
print()
print('NO TEST SET PEEKING — untouched until evaluation cell.')
print('Calibration block (2022) used ONLY for Platt scaling.')

Train  (2019-2021): 1132 rows | is_top10=0.515
Calib  (2022):       426 rows | is_top10=0.516
Test   (2023-2024):  889 rows  | is_top10=0.517

NO TEST SET PEEKING — untouched until evaluation cell.
Calibration block (2022) used ONLY for Platt scaling.


## 4 — Feature Engineering

In [6]:
TIER_MAP = {'front': 2, 'midfield': 1, 'backmarker': 0}

def build_features(split_df):
    X = pd.DataFrame(index=split_df.index)
    X['grid_position']        = split_df['grid_position'].fillna(20)
    X['constructor_tier_num'] = split_df['constructor_tier'].map(TIER_MAP).fillna(1)
    # n_stops: SCENARIO INPUT — declared in leakage audit
    X['n_stops']              = split_df['n_stops'].fillna(1)
    return X

X_train = build_features(train); y_train = train['is_top10']
X_calib = build_features(calib); y_calib = calib['is_top10']
X_test  = build_features(test);  y_test  = test['is_top10']

print('Features: grid_position [PRE-RACE] | constructor_tier_num [PRE-RACE] | n_stops [SCENARIO INPUT]')
print(f'Shapes — Train: {X_train.shape} | Calib: {X_calib.shape} | Test: {X_test.shape}')
X_train.describe().round(2)

Features: grid_position [PRE-RACE] | constructor_tier_num [PRE-RACE] | n_stops [SCENARIO INPUT]
Shapes — Train: (1132, 3) | Calib: (426, 3) | Test: (889, 3)


,grid_position,constructor_tier_num,n_stops
count,1132.00,1132.00,1132.00
mean,10.52,0.88,1.64
std,5.81,0.68,0.87
min,1.00,0.00,0.00
25%,5.00,0.00,1.00
50%,11.00,1.00,1.00
75%,16.00,1.00,2.00
max,20.00,2.00,6.00


## 5 — Baseline 1: Heuristic (F1-Defendable)

**Rationale:** La posicion en la parrilla es el predictor pre-carrera mas fuerte en F1. P1-P5 tienen ritmo para mantener top-10 en condiciones normales. P6-P10 estan dentro de la zona de puntos pero bajo presion de adelantamiento. P11+ requiere attrition significativa para puntuar. Umbrales fijados desde logica F1 y validados en datos de entrenamiento.

In [7]:
def heuristic_predict(X):
    gpos = X['grid_position']
    return np.where(gpos <= 5, 0.87, np.where(gpos <= 10, 0.68, 0.25))

print('=== Validacion de umbrales heuristicos en TRAINING SET (2019-2021) ===')
for mask, label, threshold in [
    (train['grid_position'] <= 5,  'grid <= 5   (heuristica=0.87)', 0.87),
    ((train['grid_position'] > 5) & (train['grid_position'] <= 10), 'grid 6-10  (heuristica=0.68)', 0.68),
    (train['grid_position'] > 10, 'grid > 10  (heuristica=0.25)', 0.25),
]:
    actual = train[mask]['is_top10'].mean()
    print(f'  {label}: P(top10) real = {actual:.3f}  (n={mask.sum()})')

=== Validacion de umbrales heuristicos en TRAINING SET (2019-2021) ===
  grid <= 5   (heuristica=0.87): P(top10) real = 0.874  (n=285)
  grid 6-10  (heuristica=0.68): P(top10) real = 0.679  (n=280)
  grid > 10  (heuristica=0.25): P(top10) real = 0.254  (n=567)


## 6 — Baseline 2: Logistic Regression + Platt Scaling

Entrenado en 2019-2021. Calibrado con Platt scaling en **2022 unicamente** (implementacion manual — `cv='prefit'` eliminado en sklearn >= 1.2).

In [8]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)

# Platt scaling: ajustar sigmoide sobre scores crudos del LR usando solo 2022
calib_raw = lr.predict_proba(X_calib)[:, 1].reshape(-1, 1)
platt = LogisticRegression(C=1e10, max_iter=1000)
platt.fit(calib_raw, y_calib)

def lr_predict_calibrated(X):
    raw = lr.predict_proba(X)[:, 1].reshape(-1, 1)
    return platt.predict_proba(raw)[:, 1]

print('LR entrenado en 2019-2021.')
print('Platt scaling calibrado en 2022 UNICAMENTE.')
print('Coeficientes:')
for feat, coef in zip(X_train.columns, lr.coef_[0]):
    print(f'  {feat}: {coef:.4f}')
print(f'  intercept: {lr.intercept_[0]:.4f}')

LR entrenado en 2019-2021.
Platt scaling calibrado en 2022 UNICAMENTE.
Coeficientes:
  grid_position: -0.1996
  constructor_tier_num: 0.9104
  n_stops: 0.0496
  intercept: 1.3412


## 7 — Evaluacion en Test Set (2023-2024)

> **Esta celda mira el test set exactamente una vez. El modelo queda bloqueado despues de esto.**

In [9]:
p_heuristic = heuristic_predict(X_test)
p_lr        = lr_predict_calibrated(X_test)

def report(y_true, y_prob):
    return (brier_score_loss(y_true, y_prob),
            log_loss(y_true, y_prob),
            roc_auc_score(y_true, y_prob))

bs_h, ll_h, auc_h   = report(y_test, p_heuristic)
bs_lr, ll_lr, auc_lr = report(y_test, p_lr)

print('=' * 68)
print(f'{"Modelo":<36} {"Brier":>8}  {"LogLoss":>8}  {"ROC-AUC":>8}')
print('-' * 68)
print(f'{"Docent reference  (objetivo)":<36} {0.1320:>8.4f}  {"---":>8}  {0.8920:>8.4f}')
print(f'{"Grid-rule reference (piso)":<36} {0.2080:>8.4f}  {"---":>8}  {"---":>8}')
print('-' * 68)
print(f'{"Heuristica (grid_position buckets)":<36} {bs_h:>8.4f}  {ll_h:>8.4f}  {auc_h:>8.4f}')
print(f'{"LR + Platt Calibration":<36} {bs_lr:>8.4f}  {ll_lr:>8.4f}  {auc_lr:>8.4f}')
print('=' * 68)
print(f'Supera grid-rule (0.208): {"SI" if bs_lr < 0.208 else "NO"}')
print(f'Supera docent   (0.132): {"SI" if bs_lr < 0.132 else "NO — gap objetivo de experimentos Hito 2"}')

Modelo                                  Brier   LogLoss   ROC-AUC
--------------------------------------------------------------------
Docent reference  (objetivo)           0.1320       ---    0.8920
Grid-rule reference (piso)             0.2080       ---       ---
--------------------------------------------------------------------
Heuristica (grid_position buckets)     0.1606    0.4976    0.8173
LR + Platt Calibration                 0.1396    0.4436    0.8766
Supera grid-rule (0.208): SI
Supera docent   (0.132): NO — gap objetivo de experimentos Hito 2


## 8 — Curva de Calibracion

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name, probs in [
    (axes[0], 'Heuristica (grid_position buckets)', p_heuristic),
    (axes[1], 'LR + Platt Calibration', p_lr),
]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=8, strategy='quantile')
    ax.plot(mean_pred, frac_pos, 's-', color='steelblue', linewidth=2, label=name)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Calibracion perfecta')
    bs = brier_score_loss(y_test, probs)
    ax.set_title(f'{name}\nTest 2023-2024 | Brier = {bs:.4f}')
    ax.set_xlabel('Probabilidad predicha promedio')
    ax.set_ylabel('Fraccion de positivos')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Curvas de Calibracion — Hito 1 Baselines', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('calibration_curve_test.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: calibration_curve_test.png')

Guardado: calibration_curve_test.png


/var/folders/x0/30ryv2n16ds0p85x77sdwq1r0000gn/T/ipykernel_4110/2022874806.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9 — Comparacion What-If de Escenarios

El ingeniero fija el contexto pre-carrera y varia `n_stops` para comparar estrategias.

In [11]:
def scenario_predict(grid_position, constructor_tier, n_stops):
    X = pd.DataFrame([{
        'grid_position': grid_position,
        'constructor_tier_num': TIER_MAP.get(constructor_tier, 1),
        'n_stops': n_stops,
    }])
    return lr_predict_calibrated(X)[0]

print('=== COMPARACION WHAT-IF — Baseline LR Calibrado ===')
print()
print('Escenario A — Monaco 2024, Norris (McLaren=front, grid=4)')
pA1 = scenario_predict(grid_position=4, constructor_tier='front', n_stops=1)
pA2 = scenario_predict(grid_position=4, constructor_tier='front', n_stops=2)
print(f'  A1: grid=4, front, n_stops=1, compound=M-H   → P(top10) = {pA1:.3f}')
print(f'  A2: grid=4, front, n_stops=2, compound=S-M-H → P(top10) = {pA2:.3f}')
print(f'  Delta (A1-A2): {pA1-pA2:+.3f}')
print()
print('Escenario B — Italian GP 2023, Russell (Mercedes=midfield, grid=8)')
pB1 = scenario_predict(grid_position=8, constructor_tier='midfield', n_stops=1)
pB2 = scenario_predict(grid_position=8, constructor_tier='midfield', n_stops=2)
print(f'  B1: grid=8, midfield, n_stops=1, compound=H-M   → P(top10) = {pB1:.3f}')
print(f'  B2: grid=8, midfield, n_stops=2, compound=S-H-M → P(top10) = {pB2:.3f}')
print(f'  Delta (B2-B1): {pB2-pB1:+.3f}')
print()
print('NOTA: n_stops es SCENARIO INPUT — el ingeniero lo fija deliberadamente.')
print('En Hito 2 se codificara compound_sequence para ampliar el delta entre estrategias.')


=== COMPARACION WHAT-IF — Baseline LR Calibrado ===

Escenario A — Monaco 2024, Norris (McLaren=front, grid=4)
  A1: grid=4, front, n_stops=1, compound=M-H   → P(top10) = 0.897
  A2: grid=4, front, n_stops=2, compound=S-M-H → P(top10) = 0.899
  Delta (A1-A2): -0.002

Escenario B — Italian GP 2023, Russell (Mercedes=midfield, grid=8)
  B1: grid=8, midfield, n_stops=1, compound=H-M   → P(top10) = 0.712
  B2: grid=8, midfield, n_stops=2, compound=S-H-M → P(top10) = 0.723
  Delta (B2-B1): +0.011

NOTA: n_stops es SCENARIO INPUT — el ingeniero lo fija deliberadamente.
En Hito 2 se codificara compound_sequence para ampliar el delta entre estrategias.


## 10 — Stress-Test: Slice Safety Car

`safety_car_periods` es AUDIT SLICE (resultado de carrera). NO se usa como predictor del modelo.  
Solo se usa aqui para verificar si el rendimiento del modelo difiere en carreras con SC.

In [12]:
# weather_actual es el audit slice disponible (safety_car_periods = 0 en todo el dataset)
wet_mask = test['weather_actual'] == 'wet'
dry_mask = test['weather_actual'] == 'dry'

print('=== STRESS-TEST CLIMA (LR+Platt en test set) ===')
print(f'Carreras MOJADAS (n={wet_mask.sum()}): ', end='')
if wet_mask.sum() > 0:
    print(f'Brier={brier_score_loss(y_test[wet_mask], p_lr[wet_mask]):.4f}')
else:
    print('n=0 en este split')
print(f'Carreras SECAS   (n={dry_mask.sum()}): Brier={brier_score_loss(y_test[dry_mask], p_lr[dry_mask]):.4f}')
print()
print('NOTA: safety_car_periods = 0 en todo el dataset (limitacion conocida).')
print('weather_actual es post-carrera — AUDIT SLICE, nunca predictor.')


=== STRESS-TEST CLIMA (LR+Platt en test set) ===
Carreras MOJADAS (n=121): Brier=0.1390
Carreras SECAS   (n=768): Brier=0.1397

NOTA: safety_car_periods = 0 en todo el dataset (limitacion conocida).
weather_actual es post-carrera — AUDIT SLICE, nunca predictor.


## 11 — Resumen Final

In [13]:
print('=' * 65)
print('HITO 1 RESUMEN — F1 Race Strategy Advisor Baseline')
print('=' * 65)
print(f'Dataset:   f1_strategy_race_level.csv ({len(df)} filas, 2019-2024)')
print(f'Target:    is_top10 (finish_position <= 10)')
print(f'Split:     train={len(train)} | calib={len(calib)} | test={len(test)}')
print()
print('Features del modelo baseline:')
print('  grid_position        [PRE-RACE]')
print('  constructor_tier_num [PRE-RACE] front=2 / midfield=1 / backmarker=0')
print('  n_stops              [SCENARIO INPUT]')
print()
print('Resultados en test (2023-2024):')
print(f'  Grid-rule (piso):    Brier=0.2080')
print(f'  Docent (objetivo):   Brier=0.1320  ROC-AUC=0.8920')
print(f'  Heuristica:          Brier={bs_h:.4f}  LogLoss={ll_h:.4f}  ROC-AUC={auc_h:.4f}')
print(f'  LR + Platt:          Brier={bs_lr:.4f}  LogLoss={ll_lr:.4f}  ROC-AUC={auc_lr:.4f}')
print()
print(f'  Supera grid-rule (0.208): {"SI" if bs_lr < 0.208 else "NO"}')
print(f'  Supera docent   (0.132): {"SI" if bs_lr < 0.132 else "NO"}')
print()
print('Leakage: LIMPIO — predictores PRE-RACE (grid_position, constructor_tier) + SCENARIO INPUT (n_stops)')
print('Calibracion: Platt scaling en 2022 UNICAMENTE')
print('safety_car_periods: AUDIT SLICE — no predictor')
print()
print('Experimentos Hito 2 (ver framing.md §6):')
print('  E1 — Random Forest + features pre-carrera  (objetivo Brier <= 0.160)')
print('  E2 — XGBoost + compound_sequence codificado (objetivo Brier <= 0.140)')
print('  E3 — Isotonic vs Platt: comparacion de calibracion')

HITO 1 RESUMEN — F1 Race Strategy Advisor Baseline
Dataset:   f1_strategy_race_level.csv (2447 filas, 2019-2024)
Target:    is_top10 (finish_position <= 10)
Split:     train=1132 | calib=426 | test=889

Features del modelo baseline:
  grid_position        [PRE-RACE]
  constructor_tier_num [PRE-RACE] front=2 / midfield=1 / backmarker=0
  n_stops              [SCENARIO INPUT]

Resultados en test (2023-2024):
  Grid-rule (piso):    Brier=0.2080
  Docent (objetivo):   Brier=0.1320  ROC-AUC=0.8920
  Heuristica:          Brier=0.1606  LogLoss=0.4976  ROC-AUC=0.8173
  LR + Platt:          Brier=0.1396  LogLoss=0.4436  ROC-AUC=0.8766

  Supera grid-rule (0.208): SI
  Supera docent   (0.132): NO

Leakage: LIMPIO — predictores PRE-RACE (grid_position, constructor_tier) + SCENARIO INPUT (n_stops)
Calibracion: Platt scaling en 2022 UNICAMENTE
safety_car_periods: AUDIT SLICE — no predictor

Experimentos Hito 2 (ver framing.md §6):
  E1 — Random Forest + features pre-carrera  (objetivo Brier <= 0.16